# Test-Time Compute & Reinforcement Learning: Mathematical Foundations of GRPO & DeepSeek-R1

> **Topic**: Reasoning LLMs, Group Relative Policy Optimization (GRPO), Test-Time Search  
> **Key Insight**: How to train LLMs to self-reflect and reason mathematically without a Value Network.

---

## 1. Mathematical Objective: PPO vs. GRPO
Standard Proximal Policy Optimization (PPO) trains two models:
1. An **Actor** $\pi_\theta(y|x)$ (the policy generating responses).
2. A **Critic** $V_\phi(x)$ (the value model estimating baseline expected reward).

For massive LLMs (e.g. 70B parameters), loading the Critic requires almost as much GPU memory as the Actor itself, doubling infrastructure costs.

### The GRPO Innovation
Group Relative Policy Optimization (GRPO) completely eliminates the Critic network $V_\phi$.
For each prompt $q$, GRPO samples a group of $G$ candidate outputs $\{o_1, o_2, \dots, o_G\} \sim \pi_{\theta_{\text{old}}}(q)$.
Rewards $\{r_1, r_2, \dots, r_G\}$ are computed using verifiable task rules (e.g., correct math proof = +1, format compliance = +0.1).

The advantage $\hat{A}_i$ is normalized relative to the group:
$$\hat{A}_i = \frac{r_i - \text{mean}(\{r_1, \dots, r_G\})}{\text{std}(\{r_1, \dots, r_G\}) + \epsilon}$$

The objective maximizes clipped surrogate advantage penalized by KL-divergence:
$$\mathcal{J}_{\text{GRPO}}(\theta) = \mathbb{E}_{q \sim P, \{o_i\} \sim \pi_{\theta_{\text{old}}}} \left[ \frac{1}{G} \sum_{i=1}^G \min \left( \frac{\pi_\theta(o_i|q)}{\pi_{\theta_{\text{old}}}(o_i|q)} \hat{A}_i, \, \text{clip}\left(\frac{\pi_\theta(o_i|q)}{\pi_{\theta_{\text{old}}}(o_i|q)}, 1-\epsilon, 1+\epsilon\right) \hat{A}_i \right) - \beta \mathbb{D}_{\text{KL}}(\pi_\theta \parallel \pi_{\text{ref}}) \right]$$


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

def compute_grpo_advantages(rewards, eps=1e-8):
    """Compute group relative advantages without a critic network."""
    mean_r = rewards.mean(dim=-1, keepdim=True)
    std_r = rewards.std(dim=-1, keepdim=True)
    return (rewards - mean_r) / (std_r + eps)

# Simulated group of 8 generated reasoning chains
rewards = torch.tensor([[0.0, 1.0, 0.0, 1.0, 1.0, 0.0, 0.0, 1.0]]) # 4 correct, 4 wrong
adv = compute_grpo_advantages(rewards)
print("Rewards:   ", rewards[0].tolist())
print("Advantages:", [round(x, 3) for x in adv[0].tolist()])


---
## 2. Test-Time Compute Scaling Law
Instead of scaling parameter count $N$, test-time compute scales inference tokens $T$:
$$\text{Accuracy} \propto \alpha \log(T_{\text{thinking}})$$
Through GRPO, models spontaneously learn long-horizon reasoning tokens: `<think> ... </think>`, backtracking, and error-checking without supervised demonstration!
